In [1]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
import json, os

spark = SparkSession.builder \
    .appName('debug_gold_skill') \
    .config('spark.sql.catalog.nessie', 'org.apache.iceberg.spark.SparkCatalog') \
    .config('spark.sql.catalog.nessie.ref', 'demo') \
    .config('spark.sql.catalog.nessie.uri', 'http://nessie:19120/api/v1') \
    .config('spark.sql.catalog.nessie.authentication.type', 'NONE') \
    .config('spark.sql.catalog.nessie.catalog-impl', 'org.apache.iceberg.nessie.NessieCatalog') \
    .config('spark.sql.catalog.nessie.s3.endpoint', 'http://minio:9000') \
    .config('spark.sql.catalog.nessie.warehouse', 's3://warehouse') \
    .config('spark.sql.catalog.nessie.io-impl', 'org.apache.iceberg.aws.s3.S3FileIO') \
    .config('spark.sql.catalog.nessie.s3.access-key-id', 'admin') \
    .config('spark.sql.catalog.nessie.s3.secret-access-key', 'password') \
    .config('spark.sql.catalog.nessie.s3.path-style-access', 'true') \
    .config('spark.sql.catalog.nessie.cache-enabled', 'false') \
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem') \
    .config('spark.hadoop.fs.s3a.access.key', 'admin') \
    .config('spark.hadoop.fs.s3a.secret.key', 'password') \
    .config('spark.hadoop.fs.s3a.endpoint', 'http://minio:9000') \
    .config('spark.hadoop.fs.s3a.path.style.access', 'true') \
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false') \
    .config('spark.driver.extraJavaOptions', '-Daws.region=us-east-1') \
    .config('spark.executor.extraJavaOptions', '-Daws.region=us-east-1') \
    .config('spark.sql.extensions', 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions') \
    .config('spark.sql.shuffle.partitions', '4') \
    .getOrCreate()

spark.sparkContext.setLogLevel('WARN')
print('Spark version:', spark.version)
print('✅ SparkSession OK')

Spark version: 3.5.3
✅ SparkSession OK


In [2]:
# Check xem dim_skill có skills nào match với raw skills từ silver không
df_raw_sample = df_all_raw_test.select(F.lower(F.trim(F.col("raw_skill"))).alias("raw_skill")).distinct()
dim_skill_current = spark.table("nessie.gold.dim_skill") \
    .select(F.col("skill_name").alias("canonical_skill"), "skill_key")

# Simulate df_mapped khi không có skill_alias.json
df_mapped_sim = df_raw_sample.withColumn("canonical_skill", F.col("raw_skill"))



NameError: name 'df_all_raw_test' is not defined

In [ ]:
# Join như nhánh else
matched = df_mapped_sim.join(dim_skill_current, on="canonical_skill", how="inner")
print(matched.count())

In [3]:
SILVER_TABLE = 'nessie.silver.jobs'
GOLD_NAMESPACE = 'nessie.gold'

df_silver_all = spark.table(SILVER_TABLE)
total = df_silver_all.count()
pending = df_silver_all.filter(F.col('gold_processed') == False).count()
print(f'Total silver records  : {total}')
print(f'Pending (gold=False)  : {pending}')

if pending == 0:
    print('⚠️  Không có record pending! Gold ETL sẽ exit sớm → dim_skill không được chạy.')
    print('    → Kiểm tra xem tất cả records đã bị đánh dấu gold_processed=True chưa?')
else:
    print(f'✅ Có {pending} records pending')

Total silver records  : 131367
Pending (gold=False)  : 3419
✅ Có 3419 records pending


In [3]:
df_pending = spark.table(SILVER_TABLE).filter(F.col('gold_processed') == False).limit(500)
df_pending.cache()
df_pending.count()

# Kiểm tra schema
print('=== Schema của silver ===')
for field in df_pending.schema.fields:
    if 'skill' in field.name.lower():
        print(f'  {field.name}: {field.dataType}')

print()

# Kiểm tra xem tech_skills / soft_skills null hay empty
tech_null  = df_pending.filter(F.col('tech_skills').isNull()).count()
tech_empty = df_pending.filter(F.size(F.col('tech_skills')) == 0).count()
tech_ok    = df_pending.filter(F.size(F.col('tech_skills')) > 0).count()

soft_null  = df_pending.filter(F.col('soft_skills').isNull()).count()
soft_empty = df_pending.filter(F.size(F.col('soft_skills')) == 0).count()
soft_ok    = df_pending.filter(F.size(F.col('soft_skills')) > 0).count()

print('tech_skills:')
print(f'  null   : {tech_null}')
print(f'  empty  : {tech_empty}')
print(f'  có data: {tech_ok}')
print()
print('soft_skills:')
print(f'  null   : {soft_null}')
print(f'  empty  : {soft_empty}')
print(f'  có data: {soft_ok}')

if tech_ok == 0 and soft_ok == 0:
    print()
    print('🚨 BUG FOUND: Cả tech_skills lẫn soft_skills đều rỗng/null!')
    print('   → NER model chưa chạy, hoặc silver được write TRƯỚC khi NER enrich')
    print('   → df_all_raw sẽ rỗng → ensure_skill_dim_and_alias không insert gì cả')

=== Schema của silver ===
  tech_skills: ArrayType(StringType(), True)
  soft_skills: ArrayType(StringType(), True)
  skills_all: ArrayType(StringType(), True)

tech_skills:
  null   : 0
  empty  : 188
  có data: 312

soft_skills:
  null   : 0
  empty  : 96
  có data: 404


In [4]:
# Lấy records có skill
df_with_skills = df_pending.filter(F.size(F.col('tech_skills')) > 0)

if df_with_skills.count() == 0:
    print('⚠️  Không có record nào có tech_skills. Thử xem soft_skills...')
    df_with_skills = df_pending.filter(F.size(F.col('soft_skills')) > 0)

if df_with_skills.count() == 0:
    print('🚨 Không có record nào có bất kỳ skill nào. Xem skills_all...')
    df_pending.select('link', 'skills_all', 'tech_skills', 'soft_skills').show(5, truncate=False)
else:
    print(f'Found {df_with_skills.count()} records with skills. Sample:')
    df_with_skills.select('link', 'tech_skills', 'soft_skills').show(5, truncate=False)

NameError: name 'df_pending' is not defined

In [ ]:
df_latest = df_pending  # dùng luôn batch pending để simulate

# Đây là đoạn code trong gold_demo.py:
df_tech_raw = df_latest \
    .select(F.explode('tech_skills').alias('raw_skill')) \
    .filter(F.col('raw_skill').isNotNull() & (F.trim(F.col('raw_skill')) != ''))

df_soft_raw = df_latest \
    .select(F.explode('soft_skills').alias('raw_skill')) \
    .filter(F.col('raw_skill').isNotNull() & (F.trim(F.col('raw_skill')) != ''))

df_all_raw = df_tech_raw.union(df_soft_raw).distinct()

count_all_raw = df_all_raw.count()
print(f'df_all_raw count: {count_all_raw}')

if count_all_raw == 0:
    print('🚨 BUG CONFIRMED: df_all_raw là rỗng!')
    print('   ensure_skill_dim_and_alias sẽ không insert gì vào dim_skill')
else:
    print('✅ df_all_raw có data. Sample:')
    df_all_raw.show(10, truncate=False)

In [ ]:
# Các path có thể có
CANDIDATE_PATHS = [
    '/opt/airflow/scripts/skill_alias.json',
    '/opt/bitnami/spark/scripts/skill_alias.json',
    '/home/jovyan/notebooks/skill_alias.json',
    './skill_alias.json',
]

SKILL_MAPPING_PATH = None
for p in CANDIDATE_PATHS:
    if os.path.exists(p):
        SKILL_MAPPING_PATH = p
        print(f'✅ Found skill_alias.json at: {p}')
        break

if SKILL_MAPPING_PATH is None:
    print('🚨 BUG FOUND: skill_alias.json KHÔNG TỒN TẠI ở bất kỳ path nào!')
    print('   → load_skill_mapping() sẽ raise FileNotFoundError')
    print('   → Nếu Airflow swallow exception, dim_skill sẽ trống')
    print()
    print('   FIX: Tạo file skill_alias.json hoặc dùng empty dict {} nếu chưa có mapping')
else:
    with open(SKILL_MAPPING_PATH) as f:
        data = json.load(f)
    print(f'Số entries trong skill_alias.json: {len(data)}')
    if len(data) > 0:
        sample_key = list(data.keys())[0]
        print(f'Sample entry: "{sample_key}" → {data[sample_key]}')
    
    rows = [(raw.strip().lower(), info['canonical'], info['type']) for raw, info in data.items()]
    mapping_df = spark.createDataFrame(rows, schema='raw_skill STRING, canonical_skill STRING, skill_type STRING')
    print(f'mapping_df count: {mapping_df.count()}')

In [ ]:
# Nếu df_all_raw rỗng, tạo mock data để test logic phần còn lại
if count_all_raw == 0:
    print('⚠️  df_all_raw rỗng → tạo mock skills để test phần logic phía dưới')
    mock_skills = [('python',), ('java',), ('docker',), ('communication',), ('teamwork',)]
    df_all_raw_test = spark.createDataFrame(mock_skills, ['raw_skill'])
else:
    df_all_raw_test = df_all_raw

print(f'Testing với {df_all_raw_test.count()} raw skills')

# Nếu chưa load mapping_df, tạo empty
if SKILL_MAPPING_PATH is None:
    print('Dùng empty mapping_df (không có skill_alias.json)')
    mapping_df = spark.createDataFrame([], 'raw_skill STRING, canonical_skill STRING, skill_type STRING')

In [ ]:
# STEP 6a: df_raw sau normalize
df_raw = df_all_raw_test.select(F.lower(F.trim(F.col('raw_skill'))).alias('raw_skill')).distinct()
print(f'df_raw (normalized, distinct): {df_raw.count()} rows')
df_raw.show(10, truncate=False)

In [ ]:
# STEP 6b: df_mapped sau join với mapping
df_mapped = df_raw.join(mapping_df, on='raw_skill', how='left') \
    .select(
        'raw_skill',
        F.coalesce(F.col('canonical_skill'), F.col('raw_skill')).alias('canonical_skill'),
        F.coalesce(F.col('skill_type'), F.lit('unknown')).alias('skill_type'),
    )

print(f'df_mapped count: {df_mapped.count()}')
df_mapped.show(10, truncate=False)

# Check null
null_canonical = df_mapped.filter(F.col('canonical_skill').isNull()).count()
print(f'Rows với canonical_skill NULL: {null_canonical}  (sẽ bị drop ở dim_skill)')

In [ ]:
# STEP 6c: df_canon
df_canon = df_mapped.select('canonical_skill', 'skill_type').distinct()
print(f'df_canon (distinct canonical skills): {df_canon.count()}')
df_canon.show(10, truncate=False)

In [ ]:
# STEP 6d: Check dim_skill hiện tại
existing_skills = spark.table(f'{GOLD_NAMESPACE}.dim_skill').select('skill_name', 'skill_key')
existing_count = existing_skills.count()
print(f'dim_skill hiện tại có: {existing_count} rows')
if existing_count > 0:
    existing_skills.show(10)
else:
    print('dim_skill TRỐNG → left_anti join sẽ trả về toàn bộ df_canon')

In [ ]:
# STEP 6e: df_new_canon sau left_anti
df_new_canon = df_canon.join(
    existing_skills, df_canon.canonical_skill == existing_skills.skill_name, 'left_anti'
).select('canonical_skill', 'skill_type')

new_canon_count = df_new_canon.count()
print(f'df_new_canon (skills chưa có trong dim_skill): {new_canon_count} rows')

if new_canon_count == 0 and existing_count == 0:
    print('🚨 BUG: df_new_canon rỗng dù dim_skill cũng rỗng!')
    print('   → Kiểm tra lại join condition. canonical_skill vs skill_name có match không?')
elif new_canon_count == 0:
    print('ℹ️  Tất cả skills đã tồn tại trong dim_skill → sẽ chạy nhánh else (alias only)')
else:
    print(f'✅ Có {new_canon_count} skills mới cần insert')
    df_new_canon.show(10, truncate=False)

In [ ]:
# STEP 6f: 🚨 KIỂM TRA BUG NGHIÊM TRỌNG — isEmpty() và lazy evaluation
print('=== Kiểm tra isEmpty() behavior ===')
print()
print('isEmpty() trong Spark gọi .take(1), nếu DF là lazy và có side-effect')
print('thì kết quả có thể khác với .count() > 0')
print()

# Recreate df_new_canon KHÔNG cache để test
df_new_canon_lazy = df_canon.join(
    existing_skills, df_canon.canonical_skill == existing_skills.skill_name, 'left_anti'
).select('canonical_skill', 'skill_type')

is_empty_result = df_new_canon_lazy.isEmpty()  # Đây là vấn đề trong Spark < 3.3
count_result = df_new_canon_lazy.count()

print(f'df_new_canon.isEmpty() = {is_empty_result}')
print(f'df_new_canon.count()   = {count_result}')

if is_empty_result != (count_result == 0):
    print('🚨 BUG: isEmpty() và count() KHÔNG nhất quán!')
else:
    print('✅ isEmpty() consistent với count()')

In [5]:
# STEP 6g: 🚨 KIỂM TRA BUG row_number + Window sau cache
# Đây là bug tinh tế: df_new_canon.cache() rồi mới withColumn row_number
# Nhưng trong code gốc thứ tự là: withColumn row_number TRƯỚC cache → OK
# Tuy nhiên khi join df_mapped với df_new_canon, row_number có thể bị re-evaluate

print('=== Kiểm tra row_number stability ===')

if new_canon_count > 0:
    max_key_row = spark.sql(f'SELECT COALESCE(MAX(skill_key), 0) FROM {GOLD_NAMESPACE}.dim_skill').collect()[0][0]
    print(f'max_key trong dim_skill hiện tại: {max_key_row}')

    # Simulate như code gốc
    df_nc = df_new_canon.withColumn(
        'skill_key',
        F.row_number().over(Window.orderBy('canonical_skill')) + max_key_row
    )
    df_nc.cache()
    materialized_count = df_nc.count()  # force materialization

    print(f'Sau cache + count: {materialized_count} rows')
    df_nc.show(10)

    # Kiểm tra join với df_mapped để write alias
    df_alias_to_write = df_mapped.join(df_nc, on='canonical_skill', how='inner') \
        .select('raw_skill', 'skill_key')
    alias_count = df_alias_to_write.count()
    print(f'dim_skill_alias sẽ được insert: {alias_count} rows')
    df_alias_to_write.show(10)

    df_nc.unpersist()
else:
    print('Bỏ qua vì df_new_canon rỗng')

=== Kiểm tra row_number stability ===


NameError: name 'new_canon_count' is not defined

In [15]:
alias_current = spark.table(f'{GOLD_NAMESPACE}.dim_skill_alias')
alias_count = alias_current.count()
print(f'dim_skill_alias hiện tại: {alias_count} rows')
if alias_count > 0:
    alias_current.show(10)

# Kiểm tra orphan alias (alias trỏ đến skill_key không tồn tại trong dim_skill)
if alias_count > 0 and existing_count > 0:
    orphan = alias_current.join(
        spark.table(f'{GOLD_NAMESPACE}.dim_skill').select('skill_key'),
        on='skill_key',
        how='left_anti'
    ).count()
    print(f'Orphan aliases (skill_key không có trong dim_skill): {orphan}')
    if orphan > 0:
        print('🚨 BUG: Có alias trỏ đến skill không tồn tại!')

dim_skill_alias hiện tại: 0 rows


In [16]:
fjs = spark.table(f'{GOLD_NAMESPACE}.fact_job_skills')
fjs_count = fjs.count()
print(f'fact_job_skills: {fjs_count} rows')

# Simulate refresh_fact_job_skills để xem tại sao không có record
print()
print('=== Simulate refresh_fact_job_skills ===')

df_for_skills = df_pending.withColumn('job_key', F.col('link'))

df_tech = df_for_skills.select('job_key', F.explode('tech_skills').alias('raw_skill')).withColumn('original_type', F.lit('tech'))
df_soft = df_for_skills.select('job_key', F.explode('soft_skills').alias('raw_skill')).withColumn('original_type', F.lit('soft'))
df_skills = df_tech.union(df_soft) \
    .filter(F.col('raw_skill').isNotNull() & (F.trim(F.col('raw_skill')) != ''))

print(f'Skills exploded từ pending jobs: {df_skills.count()}')

if df_skills.count() > 0:
    df_skills = df_skills.withColumn('raw_skill_norm', F.lower(F.trim(F.col('raw_skill'))))
    alias_df = spark.table(f'{GOLD_NAMESPACE}.dim_skill_alias').withColumnRenamed('raw_skill', 'alias_raw')
    df_mapped_fjs = df_skills.join(alias_df, df_skills.raw_skill_norm == alias_df.alias_raw, 'inner') \
        .select('job_key', 'skill_key')
    
    mapped_count = df_mapped_fjs.count()
    print(f'Sau join với dim_skill_alias: {mapped_count} rows')
    
    if mapped_count == 0:
        print('🚨 BUG: Skills không match được với dim_skill_alias!')
        print('   → dim_skill_alias trống, hoặc raw_skill format không khớp')
        print()
        print('Sample raw_skill_norm từ jobs:')
        df_skills.select('raw_skill_norm').distinct().show(10, truncate=False)
        print('Sample raw_skill từ dim_skill_alias:')
        alias_df.select('alias_raw').show(10, truncate=False) if alias_df.count() > 0 else print('alias_df rỗng!')
else:
    print('⚠️  Không có skill nào để map (tech_skills + soft_skills đều rỗng)')

fact_job_skills: 0 rows

=== Simulate refresh_fact_job_skills ===
Skills exploded từ pending jobs: 2831
Sau join với dim_skill_alias: 0 rows
🚨 BUG: Skills không match được với dim_skill_alias!
   → dim_skill_alias trống, hoặc raw_skill format không khớp

Sample raw_skill_norm từ jobs:
+-------------------------------------+
|raw_skill_norm                       |
+-------------------------------------+
|an toàn pccc                         |
|thuế gtgt dịch vụ du lịch            |
|luật thuế                            |
|kỷ luật                              |
|thiết kế các sản phẩm cơ khí nội thất|
|hệ thống thông gió                   |
|phòng cháy chữa cháy                 |
|ai chat gpt                          |
|thiết                                |
|sáng tạo nội dung                    |
+-------------------------------------+
only showing top 10 rows

Sample raw_skill từ dim_skill_alias:
alias_df rỗng!


In [17]:
def ensure_skill_dim_and_alias_fixed(df_raw_skills, spark, mapping_df):
    """
    Version đã fix:
    - Cache df_mapped vì dùng 2 lần
    - Dùng count() > 0 thay isEmpty()
    - Alias toàn bộ column để tránh ambiguous
    - Materialize df_new_canon trước khi join để row_number stable
    """
    df_raw = df_raw_skills.select(F.lower(F.trim(F.col('raw_skill'))).alias('raw_skill')).distinct()
    
    # FIX: cache df_mapped vì dùng cho cả df_canon và alias write
    df_mapped = df_raw.join(mapping_df, on='raw_skill', how='left') \
        .select(
            F.col('raw_skill'),
            F.coalesce(F.col('canonical_skill'), F.col('raw_skill')).alias('canonical_skill'),
            F.coalesce(F.col('skill_type'), F.lit('unknown')).alias('skill_type'),
        )
    df_mapped.cache()
    df_mapped.count()  # materialize
    
    try:
        df_canon = df_mapped.select('canonical_skill', 'skill_type').distinct()

        existing_skills = spark.table(f'{GOLD_NAMESPACE}.dim_skill') \
            .select(
                F.col('skill_name').alias('existing_name'),  # FIX: alias để tránh ambiguous
                F.col('skill_key').alias('existing_key')
            )
        
        df_new_canon = df_canon.join(
            existing_skills,
            df_canon.canonical_skill == existing_skills.existing_name,
            'left_anti'
        ).select('canonical_skill', 'skill_type')
        
        # FIX: count() thay vì isEmpty() cho Iceberg compatibility
        new_count = df_new_canon.count()
        print(f'  New canonical skills to insert: {new_count}')

        if new_count > 0:
            max_key = spark.sql(
                f"SELECT COALESCE(MAX(skill_key), 0) FROM {GOLD_NAMESPACE}.dim_skill"
            ).collect()[0][0]

            # FIX: withColumn TRƯỚC cache, rồi force materialize
            df_new_canon_keyed = df_new_canon.withColumn(
                'skill_key',
                F.row_number().over(Window.orderBy('canonical_skill')) + max_key,
            )
            df_new_canon_keyed.cache()
            df_new_canon_keyed.count()  # materialize để row_number stable

            try:
                # Write dim_skill
                df_new_canon_keyed.select(
                    F.col('skill_key'),
                    F.col('canonical_skill').alias('skill_name'),
                    'skill_type',
                ).write.mode('append').saveAsTable(f'{GOLD_NAMESPACE}.dim_skill')
                print(f'  ✅ Inserted {new_count} rows vào dim_skill')

                # Write alias: join df_mapped với df_new_canon_keyed (đã cached)
                df_alias = df_mapped.join(df_new_canon_keyed, on='canonical_skill', how='inner') \
                    .select('raw_skill', 'skill_key')
                alias_count = df_alias.count()
                df_alias.write.mode('append').saveAsTable(f'{GOLD_NAMESPACE}.dim_skill_alias')
                print(f'  ✅ Inserted {alias_count} rows vào dim_skill_alias')
            finally:
                df_new_canon_keyed.unpersist()

        else:
            # Tất cả canonical skills đã tồn tại, chỉ cần insert alias mới
            existing_alias = spark.table(f'{GOLD_NAMESPACE}.dim_skill_alias') \
                .select(F.col('raw_skill').alias('existing_alias'))  # FIX: alias

            dim_skill_current = spark.table(f'{GOLD_NAMESPACE}.dim_skill') \
                .select(
                    F.col('skill_name').alias('canonical_skill'),
                    F.col('skill_key'),
                )

            df_new_alias = (
                df_raw
                .join(existing_alias, df_raw.raw_skill == existing_alias.existing_alias, 'left_anti')  # FIX: explicit join condition
                .join(df_mapped.select('raw_skill', 'canonical_skill'), on='raw_skill', how='inner')
                .join(dim_skill_current, on='canonical_skill', how='inner')
                .select('raw_skill', 'skill_key')
            )
            new_alias_count = df_new_alias.count()
            if new_alias_count > 0:
                df_new_alias.write.mode('append').saveAsTable(f'{GOLD_NAMESPACE}.dim_skill_alias')
                print(f'  ✅ Inserted {new_alias_count} new alias rows')
            else:
                print('  ℹ️  Không có alias mới cần insert')
    finally:
        df_mapped.unpersist()

print('Function defined. Chạy cell tiếp theo để test.')

Function defined. Chạy cell tiếp theo để test.


In [18]:
# DRY RUN — kiểm tra trước khi thực sự write
# Chỉ chạy cell này nếu bạn muốn TEST WRITE vào dim_skill
# Uncomment dòng cuối để chạy thật

print('=== DRY RUN: ensure_skill_dim_and_alias_fixed ===')
print()

if count_all_raw > 0:
    print(f'Input: {count_all_raw} raw skills từ silver pending')
    # ensure_skill_dim_and_alias_fixed(df_all_raw_test, spark, mapping_df)  # ← uncomment để chạy
    print('⚠️  Uncommend dòng trên để chạy thật (sẽ write vào dim_skill)')
else:
    print('⚠️  df_all_raw rỗng → root cause là silver không có skills, cần fix silver ETL trước')
    print()
    print('Chạy với mock data...')
    mock_skills = [('python',), ('java',), ('docker',), ('communication',), ('teamwork',)]
    df_mock = spark.createDataFrame(mock_skills, ['raw_skill'])
    # ensure_skill_dim_and_alias_fixed(df_mock, spark, mapping_df)  # ← uncomment để test với mock

=== DRY RUN: ensure_skill_dim_and_alias_fixed ===

Input: 1433 raw skills từ silver pending
⚠️  Uncommend dòng trên để chạy thật (sẽ write vào dim_skill)


In [19]:
print('=' * 60)
print('SUMMARY — ROOT CAUSE CANDIDATES')
print('=' * 60)
print()
print('1. Silver records rỗng skills?          pending =', pending, '| tech_ok =', tech_ok if 'tech_ok' in dir() else 'N/A')
print('2. df_all_raw rỗng?                     count =', count_all_raw)
print('3. skill_alias.json missing?            path =', SKILL_MAPPING_PATH)
print('4. dim_skill đã có data?                rows =', existing_count)
print('5. df_new_canon sau left_anti?          rows =', new_canon_count if 'new_canon_count' in dir() else 'N/A')
print()
print('=== LIKELY FIX ===')
if count_all_raw == 0:
    print('→ Root cause: NER model KHÔNG chạy trong silver ETL')
    print('  tech_skills và soft_skills luôn là array rỗng')
    print('  Check: jobs_demo_v1.py → enrich_with_ner() có được gọi trước append_to_silver() không?')
elif SKILL_MAPPING_PATH is None:
    print('→ Root cause: skill_alias.json không tồn tại → exception → ETL fail silently')
    print('  Fix: Mount file đúng path trong docker-compose, hoặc add fallback trong load_skill_mapping()')
elif new_canon_count == 0 and existing_count == 0:
    print('→ Root cause: left_anti join logic sai')
    print('  Fix: Dùng version ensure_skill_dim_and_alias_fixed() ở cell 9')
else:
    print('→ Cần xem thêm log Airflow để xác định exception cụ thể')

SUMMARY — ROOT CAUSE CANDIDATES

1. Silver records rỗng skills?          pending = 3419 | tech_ok = 312
2. df_all_raw rỗng?                     count = 1433
3. skill_alias.json missing?            path = None
4. dim_skill đã có data?                rows = 37951
5. df_new_canon sau left_anti?          rows = 1343

=== LIKELY FIX ===
→ Root cause: skill_alias.json không tồn tại → exception → ETL fail silently
  Fix: Mount file đúng path trong docker-compose, hoặc add fallback trong load_skill_mapping()


In [20]:
# df_pending.unpersist()
# spark.stop()
# print('Done.')

Done.
